In [146]:

import numpy as N
import pandas as pd
from dateutil import parser


1.extract 


In [ ]:
data_df=pd.read_csv("d:\project1\cyiIpFyXp6sXqdW4kxdcxQyqZ8SbPuNW.csv",encoding='utf-8-sig')
data_df

2.transform


In [ ]:
#date format 
def fix_date_january(x):
    x = str(x).strip()
    if x.lower() in ['nan', '', 'none']:
        return pd.NaT
    
    try:
        dt = parser.parse(x, dayfirst=True)
        if dt.month != 1:
            dt = dt.replace(month=dt.day, day=dt.month)
            
        return dt
    except:
        return pd.to_datetime(x, errors='coerce')

data_df['Order_Date'] = data_df['Order_Date'].apply(fix_date_january).dt.strftime('%Y-%m-%d')

#price nulls & category handling 
price=float("200")
arabic_pattern = r'[\u0600-\u06FF]+'
data_df['Category'] = data_df['Category'].str.replace(arabic_pattern, '', regex=True).replace('', np.nan)
data_df.fillna({
    'Category':"Electronics",
    "Price":price,
    "Payment_Method":"Cash"
},inplace= True)
#Quantity nulls &handling 
data_df['Quantity'] = pd.to_numeric(data_df['Quantity'], errors='coerce').fillna(1).astype(int)
#city handling and deformation
data_df["City"]=data_df["City"].str.upper()
data_df["City"]=data_df["City"].replace("ALEXANDRIA","ALEX")
#duplicates
cols=data_df.columns.drop("Order_ID")
data_df=data_df.drop_duplicates(subset=cols)
data_df=data_df.drop(19)
data_df=data_df.reset_index(drop= True)
#data types
data_df['Order_Date'] = pd.to_datetime(data_df['Order_Date'])
data_df['Order_ID'] = data_df['Order_ID'].astype(int)
data_df['Quantity'] = data_df['Quantity'].astype(int)
data_df['Price'] = data_df['Price'].astype(float)

text_columns = ['Customer_Name', 'City', 'Product', 'Category', 'Payment_Method', 'Order_Status']
for col in text_columns:
    data_df[col] = data_df[col].astype(str)
data_df['Total_Amount']=data_df['Price']*data_df["Quantity"]
data_df['Total_Amount'] = data_df['Total_Amount'].astype(float)




3. load

In [ ]:
import pyodbc
import pandas as pd

data_df['Product'] = data_df['Product'].astype(str).str.strip()
data_df['Customer_Name'] = data_df['Customer_Name'].astype(str).str.strip()
data_df['City'] = data_df['City'].astype(str).str.strip()
data_df['Total_Amount'] = data_df['Price'] * data_df['Quantity']

conn_str = r'DRIVER={ODBC Driver 17 for SQL Server};SERVER=.\SQLEXPRESS;DATABASE=SalesDB;Trusted_Connection=yes;'
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

try:
    
    products = data_df[['Product', 'Category', 'Price']].drop_duplicates()
    for _, row in products.iterrows():
        cursor.execute("""
            IF NOT EXISTS (SELECT 1 FROM Dim_Products WHERE Product_Name = ?)
            INSERT INTO Dim_Products (Product_Name, Category, Price) VALUES (?, ?, ?)
        """, row['Product'], row['Product'], row['Category'], row['Price'])
    
    customers = data_df[['Customer_Name', 'City']].drop_duplicates()
    for _, row in customers.iterrows():
        cursor.execute("""
            IF NOT EXISTS (SELECT 1 FROM Dim_Customers WHERE Customer_Name = ? AND City = ?)
            INSERT INTO Dim_Customers (Customer_Name, City) VALUES (?, ?)
        """, row['Customer_Name'], row['City'], row['Customer_Name'], row['City'])
    
    conn.commit()

    for _, row in data_df.iterrows():
        cursor.execute("SELECT Product_ID FROM Dim_Products WHERE Product_Name = ?", row['Product'])
        p_id = cursor.fetchone()[0]
        
        cursor.execute("SELECT Customer_ID FROM Dim_Customers WHERE Customer_Name = ? AND City = ?", 
                       row['Customer_Name'], row['City'])
        c_id = cursor.fetchone()[0]
        cursor.execute("""
            INSERT INTO Fact_Sales (
                Order_ID, Order_Date, Product_ID, Customer_ID, 
                Price, Quantity, Total_Amount, Payment_Method, Order_Status
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, 
        row['Order_ID'], row['Order_Date'].date(), p_id, c_id, 
        row['Price'], row['Quantity'], row['Total_Amount'], row['Payment_Method'], row['Order_Status'])

    conn.commit()
    

except Exception as e:
    conn.rollback()
finally:
    cursor.close()
    conn.close()